# 02 - Silver: Transacties

Bouwt `silver.transacties` vanuit `bronze.transacties_ing` via `silver.run_silver()` — zie `src/pipeline/transacties/silver.py`.

Stappen:
1. Dedupliceren op `rij_hash` (oudste `ingelezen_op` wint) — lost overlap tussen bestanden op
2. Type-casting: datum (`YYYYMMDD`) -> `DATE`, bedrag -> signed `DECIMAL(18,2)`
3. Opschonen van tekstvelden (trim, rekeningnummers uppercasen)
4. Stabiele surrogaatsleutel `transactie_id` op basis van `rij_hash`
5. Volledige rebuild (`CREATE OR REPLACE`) bij elke run — idempotent, past bij nightly batch


In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="requirements.txt") -> Path:
    p = Path.cwd().resolve()
    for kandidaat in [p, *p.parents]:
        if (kandidaat / marker).exists():
            return kandidaat
    raise RuntimeError("Kon project root niet vinden")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from src.pipeline.paths import DB_PAD
from src.pipeline.transacties import silver

con = duckdb.connect(str(DB_PAD))
print(f"Verbonden met {DB_PAD}")

## 1. Sanity check: bronze

Even kijken hoeveel rijen er in bronze staan en of er inderdaad duplicaten op `rij_hash` zitten (verwacht na overlappende imports).

In [ ]:
print("Totaal aantal rijen in bronze:")
print(con.execute("SELECT COUNT(*) AS aantal FROM bronze.transacties_ing").df())

print("\nAantal unieke rij_hash waarden:")
print(con.execute("SELECT COUNT(DISTINCT rij_hash) AS unieke_rijen FROM bronze.transacties_ing").df())

print("\nVoorbeeld van duplicaten (indien aanwezig):")
display(con.execute("""
    SELECT rij_hash, COUNT(*) AS aantal, array_agg(bronbestand) AS bronbestanden
    FROM bronze.transacties_ing
    GROUP BY rij_hash
    HAVING COUNT(*) > 1
    LIMIT 10
""").df())


## 2. Silver tabel bouwen

- Dedup op `rij_hash`: per groep pakken we de rij met de oudste `ingelezen_op` (`ROW_NUMBER() OVER (PARTITION BY rij_hash ORDER BY ingelezen_op ASC)`)
- `Datum` (formaat `YYYYMMDD` in ING-export) -> `DATE`
- Bedrag: `Bedrag (EUR)` (Europese notatie met komma) + `Af Bij` (Bij/Af) -> signed `DECIMAL(18,2)`
- `transactie_id`: `md5(rij_hash)` als stabiele surrogaatsleutel

Pas de datum-/bedragformaten in `silver.py` aan als jouw ING-export een ander format gebruikt (check dit met de sanity check hierboven als de cast faalt).

In [ ]:
resultaat = silver.run_silver(con)
print(resultaat)

## 3. Verificatie

In [ ]:
print("Aantal rijen in silver.transacties:")
print(con.execute("SELECT COUNT(*) AS aantal FROM silver.transacties").df())

print("\nControle: som van bedrag_eur zou moeten kloppen met wat je verwacht")
print(con.execute("SELECT SUM(bedrag_eur) AS totaal FROM silver.transacties").df())

print("\nSteekproef:")
display(con.execute("SELECT * FROM silver.transacties ORDER BY datum DESC LIMIT 10").df())

print("\nSchema:")
display(con.execute("DESCRIBE silver.transacties").df())


## 4. Controle op mislukte casts

Als een datum of bedrag niet volgens het verwachte format is, geeft `strptime`/`CAST` een fout bij het bouwen van de tabel (fail-fast, zoals bij bronze). Mocht je toch stilzwijgende `NULL`s willen opsporen na een format-wijziging, gebruik dan deze check:

In [ ]:
print("Rijen met NULL datum of bedrag na casting (zou 0 moeten zijn):")
display(con.execute("""
    SELECT * FROM silver.transacties
    WHERE datum IS NULL OR bedrag_eur IS NULL
""").df())


In [ ]:
con.close()
